# Controlled Turn-Order Experiment

**Purpose.** Estimate turn-order effects without confounding them with player
index or the policy's own first-player choice.

**Decision question.** Holding the promoted policy, deck, and opponent fixed,
does the candidate perform differently when forced to go first versus second?

This notebook changes only the `IS_FIRST` selection made by player 0. All later
decisions are delegated to the frozen promoted or official random policy.

## 1. Balanced factorial design

The observational run showed that the promoted policy chose first whenever it
occupied player 0. Seat balance alone therefore could not identify turn-order
effects. This follow-up crosses candidate seat `{0, 1}` with forced candidate
turn order `{first, second}`, producing four equal cells.

Only one loss replay per cell is retained, preventing early games from using the
entire replay quota.

In [ ]:
from collections import Counter
from copy import deepcopy
from pathlib import Path
import importlib.util
import json
import random
import shutil
import sys
import time

import numpy as np
import pandas as pd

BASE_SEED = 20260622
GAMES_PER_CELL = 10
MAX_DECISIONS = 5_000
BOOTSTRAP_SAMPLES = 10_000
REPLAYS_PER_CELL = 1
WORK_DIR = Path("/kaggle/working/controlled_turn_order_runtime")
REPLAY_DIR = Path("/kaggle/working/replays")

def first_match(pattern: str) -> Path:
    matches = sorted(Path("/kaggle/input").rglob(pattern))
    if not matches:
        raise FileNotFoundError(f"No Kaggle input matched {pattern}")
    return matches[0]

def load_module(name: str, path: Path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

sample_dir = first_match("sample_submission/main.py").parent
candidates = [
    path.parent for path in sorted(Path("/kaggle/input").rglob("main.py"))
    if "sample_submission" not in path.parts and "cg" not in path.parts
]
agent_dir = next(
    (path for path in candidates
     if (path / "main.py").exists() and (path / "deck.csv").exists()),
    None,
)
if agent_dir is None:
    raise FileNotFoundError("Attach the private agent-source dataset.")
print(f"Promoted agent: {agent_dir / 'main.py'}")
print(f"Official control: {sample_dir / 'main.py'}")

## 2. Isolated simulator and frozen policies

Copy the complete official runtime to working storage and load both policies
separately. Both players use the promoted deck, isolating policy behavior.

In [ ]:
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
shutil.copytree(sample_dir, WORK_DIR)
shutil.copy2(agent_dir / "main.py", WORK_DIR / "candidate_main.py")
shutil.copy2(agent_dir / "deck.csv", WORK_DIR / "deck.csv")
REPLAY_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(WORK_DIR))
from cg.api import OptionType, SelectContext, to_observation_class
from cg.game import battle_finish, battle_select, battle_start, visualize_data

candidate = load_module("promoted_candidate", WORK_DIR / "candidate_main.py")
control = load_module("official_random_control", sample_dir / "main.py")
deck = candidate.read_deck_csv()
assert len(deck) == 60

## 3. Controlled first-player choice and runner

`ForcedFirstChoice` intercepts only `SelectContext.IS_FIRST`. The wrapper
selects `YES` or `NO` for player 0, then delegates every subsequent observation
to the frozen base policy. Contract validation, actual `state.firstPlayer`
recording, named telemetry, and replay serialization remain unchanged.

In [ ]:
def enum_name(enum_class, value) -> str:
    try:
        return enum_class(value).name
    except (ValueError, TypeError):
        return f"UNKNOWN_{value}"

def validate_action(obs, action: list[int]) -> None:
    select = obs.select
    assert isinstance(action, list)
    assert all(isinstance(index, int) for index in action)
    assert len(action) == len(set(action))
    assert select.minCount <= len(action) <= select.maxCount
    assert all(0 <= index < len(select.option) for index in action)

def replay_observation(obs_dict: dict) -> dict:
    saved = deepcopy(obs_dict)
    saved.pop("search_begin_input", None)
    return saved

def write_replay(obs_log: list, action_log: list, path: Path) -> None:
    visual = json.loads(visualize_data())
    for index, step in enumerate(visual):
        if index < len(obs_log):
            step["obs"] = obs_log[index]
        if index < len(action_log):
            step["action"] = [action_log[index], action_log[index]]
    path.write_text(json.dumps(visual), encoding="utf-8")

replay_paths = []
replay_counts = Counter()

class ForcedFirstChoice:
    def __init__(self, base_policy, player_zero_goes_first: bool):
        self.base_policy = base_policy
        self.player_zero_goes_first = player_zero_goes_first

    def agent(self, obs_dict: dict) -> list[int]:
        obs = to_observation_class(obs_dict)
        if obs.select is not None and obs.select.context == SelectContext.IS_FIRST:
            desired = OptionType.YES if self.player_zero_goes_first else OptionType.NO
            matches = [
                index for index, option in enumerate(obs.select.option)
                if option.type == desired
            ]
            if not matches:
                raise ValueError(f"IS_FIRST does not expose {desired.name}")
            return [matches[0]]
        return self.base_policy.agent(obs_dict)

def play_game(policies: dict, game_id: int, candidate_player: int) -> dict:
    random.seed(BASE_SEED + game_id)
    started = time.perf_counter()
    decisions, first_player = 0, None
    contexts, candidate_actions = Counter(), Counter()
    obs_log, action_log = [""], [None]
    try:
        obs_dict, start_data = battle_start(deck, deck)
        if obs_dict is None:
            return {
                "status": "start_error", "game": game_id,
                "candidate_player": candidate_player,
                "error_player": start_data.errorPlayer,
                "error_type": start_data.errorType,
            }

        while decisions < MAX_DECISIONS:
            obs = to_observation_class(obs_dict)
            observed_first = getattr(obs.current, "firstPlayer", None)
            if observed_first in (0, 1):
                first_player = int(observed_first)

            if obs.current.result != -1:
                winner = int(obs.current.result)
                score = (1.0 if winner == candidate_player
                         else 0.0 if winner in (0, 1) else 0.5)
                replay_path = None
                replay_key = (candidate_player, first_player == candidate_player)
                if score == 0.0 and replay_counts[replay_key] < REPLAYS_PER_CELL:
                    replay_path = REPLAY_DIR / (
                        f"game_{game_id}_seat_{candidate_player}_"
                        f"first_{first_player}_score_{score:.1f}.json"
                    )
                    write_replay(obs_log, action_log, replay_path)
                    replay_paths.append(str(replay_path))
                    replay_counts[replay_key] += 1
                return {
                    "status": "finished", "game": game_id,
                    "candidate_player": candidate_player,
                    "first_player": first_player,
                    "candidate_went_first": first_player == candidate_player,
                    "winner": winner, "candidate_score": score,
                    "turn": int(obs.current.turn), "decisions": decisions,
                    "seconds": time.perf_counter() - started,
                    "contexts": dict(contexts),
                    "candidate_actions": dict(candidate_actions),
                    "replay_path": str(replay_path) if replay_path else None,
                }

            player = int(obs.current.yourIndex)
            contexts[enum_name(SelectContext, obs.select.context)] += 1
            action = policies[player].agent(obs_dict)
            validate_action(obs, action)
            if player == candidate_player:
                candidate_actions.update(
                    enum_name(OptionType, obs.select.option[index].type)
                    for index in action
                )
            obs_log.append(replay_observation(obs_dict))
            action_log.append(list(action))
            obs_dict = battle_select(action)
            decisions += 1

        return {
            "status": "decision_limit", "game": game_id,
            "candidate_player": candidate_player,
            "first_player": first_player, "decisions": decisions,
        }
    except Exception as error:
        return {
            "status": "exception", "game": game_id,
            "candidate_player": candidate_player,
            "first_player": first_player,
            "error": f"{type(error).__name__}: {error}",
            "decisions": decisions,
        }
    finally:
        try:
            battle_finish()
        except Exception:
            pass

## 4. Controlled 2x2 tournament

For a candidate in seat 0, forcing player 0 to choose `YES` makes the candidate
go first. For a candidate in seat 1, player 0 is the control, so its choice is
reversed. The post-run assertion verifies the simulator realized every intended
turn order.

In [ ]:
results = []
game_id = 70_000
for candidate_player in (0, 1):
    for candidate_should_go_first in (False, True):
        for repetition in range(GAMES_PER_CELL):
            player_zero_goes_first = (
                candidate_should_go_first
                if candidate_player == 0
                else not candidate_should_go_first
            )
            if candidate_player == 0:
                policies = {
                    0: ForcedFirstChoice(candidate, player_zero_goes_first),
                    1: control,
                }
            else:
                policies = {
                    0: ForcedFirstChoice(control, player_zero_goes_first),
                    1: candidate,
                }
            result = play_game(policies, game_id, candidate_player)
            result["forced_candidate_went_first"] = candidate_should_go_first
            results.append(result)
            game_id += 1

results_df = pd.DataFrame(results)
failures = results_df[results_df.status != "finished"]
finished = results_df[results_df.status == "finished"].copy()
assert failures.empty, failures.to_dict("records")
assert finished.first_player.isin([0, 1]).all(), "Missing first-player attribution"
assert (finished.candidate_went_first == finished.forced_candidate_went_first).all()

cell_counts = finished.groupby(
    ["candidate_player", "candidate_went_first"]
).size()
assert (cell_counts == GAMES_PER_CELL).all(), cell_counts.to_dict()
display(finished.drop(columns=["contexts", "candidate_actions"], errors="ignore"))
print(f"Completed {len(finished)} games; saved {len(replay_paths)} replays.")

## 5. Outcome uncertainty and controlled attribution

The overall interval is a regression screen. The primary comparison is the
candidate's score rate when forced first versus forced second. The joint table
checks that the direction is not merely a seat artifact. With ten games per
cell, treat differences as screening evidence rather than a final estimate.

In [ ]:
scores = finished.candidate_score.to_numpy(dtype=float)
rng = np.random.default_rng(BASE_SEED)
boot = rng.choice(scores, size=(BOOTSTRAP_SAMPLES, len(scores)), replace=True).mean(axis=1)
summary = {
    "games": len(finished),
    "wins": int((scores == 1.0).sum()),
    "draws": int((scores == 0.5).sum()),
    "losses": int((scores == 0.0).sum()),
    "score_rate": float(scores.mean()),
    "bootstrap_95_low": float(np.quantile(boot, 0.025)),
    "bootstrap_95_high": float(np.quantile(boot, 0.975)),
    "failures": len(failures), "replays_saved": len(replay_paths),
}
display(pd.Series(summary).to_frame("value"))

by_seat = finished.groupby("candidate_player").candidate_score.agg(
    games="size", score_rate="mean"
)
by_turn_order = finished.groupby("candidate_went_first").candidate_score.agg(
    games="size", score_rate="mean"
).rename(index={False: "went_second", True: "went_first"})
joint = finished.pivot_table(
    index="candidate_player", columns="candidate_went_first",
    values="candidate_score", aggfunc=["size", "mean"], fill_value=0,
)
display(by_seat)
display(by_turn_order)
display(joint)

## 6. Action telemetry and replay manifest

Replay files are diagnostic samples, not independent evaluation evidence.
The optional author viewer uploads JSON externally; inspect the artifact and
competition rules before using that service.

In [ ]:
action_counts, context_counts = Counter(), Counter()
for result in results:
    action_counts.update(result.get("candidate_actions", {}))
    context_counts.update(result.get("contexts", {}))
display(pd.Series(action_counts, name="candidate_selections").sort_values(ascending=False).to_frame())
display(pd.Series(context_counts, name="decisions").sort_values(ascending=False).to_frame())
replay_manifest = finished.loc[
    finished.replay_path.notna(),
    ["game", "candidate_player", "first_player", "candidate_score", "replay_path"],
]
display(replay_manifest)

In [ ]:
payload = {
    "experiment": "controlled candidate seat by turn-order factorial",
    "configuration": {
        "base_seed": BASE_SEED, "games_per_cell": GAMES_PER_CELL,
        "replays_per_cell": REPLAYS_PER_CELL, "simulator_seed_exposed": False,
    },
    "summary": summary,
    "by_seat": by_seat.reset_index().to_dict("records"),
    "by_turn_order": by_turn_order.reset_index().to_dict("records"),
    "replays": replay_manifest.to_dict("records"),
    "results": results,
}
evidence_path = Path("/kaggle/working/controlled_turn_order_experiment.json")
evidence_path.write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")
print(f"Saved evidence to {evidence_path}")

## 7. Interpretation gate

This experiment still does not promote a policy. If a turn-order difference is
consistent across both seats, make setup and attack planning explicitly aware
of `state.firstPlayer`. If the difference is weak or reverses by seat, prioritize
the deck-specific attack planner and stronger opponents instead.